### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="5g_energy_consumption",
    dataset_year="2023",
    domain_str="technology & internet",
    # Data Source
    dataset_source="HuggingFace",
    original_dataset_source_download_link="https://huggingface.co/datasets/netop/5G-Network-Energy-Consumption",
    download_description=r"""
    We use the dataset version uploaded by a top solution from the Zindi challenge.     

    mkdir -p local-data-warehouse/5g_energy_consumption/
    wget https://github.com/ITU-AI-ML-in-5G-Challenge/5G-Energy-Consumption-Modelling-Solution-Team-Farzi-Data-Scientists/raw/refs/heads/main/ITU-5G-energy-Consumption-Dataset.zip
    unzip ITU-5G-energy-Consumption-Dataset.zip -d local-data-warehouse/5g_energy_consumption/ 
    rm ITU-5G-energy-Consumption-Dataset.zip

""",
    # References
    academic_reference_bibtex=r"""@misc{huawei_netop_5g_energy_consumption,
  author       = {{HUAWEI Netop Team}},
  title        = {5G Network Energy Consumption Dataset},
  year         = {n.d.},
  howpublished = {\url{https://huggingface.co/datasets/netop/5G-Network-Energy-Consumption}},
  note         = {Dataset hosted on Hugging Face, accessed 2026-04-15}
}
""",
    academic_reference_bibtex_key="huawei_netop_5g_energy_consumption",
    license="MIT",
    data_tags=["Non-IID", "Grouped", "Temporal"],
    curation_comments="""
    - Note: The dataset was used in a Zindi challenge, but also uploaded to Huggingface under a MIT license by the company (Huawei).
    - The corresponding ITU/Zindi challenge explicitly emphasizes generalization to unseen base station products/configurations. We therefore split by base_station (BS).
    - With this setup, the task is development of predictive models for network optimization where models need to generalize to new unseen base station configurations and estimate their energy consumption under similar conditions (during the same time period).
    - For preprocessing, we orient on a top solution from the Zindi challenge: https://github.com/ITU-AI-ML-in-5G-Challenge/5G-Energy-Consumption-Modelling-Solution-Team-Farzi-Data-Scientists/tree/main.
    - Note that the competition also used mostly samples from the known BS as the test set, but weighted unknown base stations higher in evaluation. 
    - The data itself is time-series. However, because we predict entirely unseen base stations in a per-sample fashion, the row-wise dependencies resulting from the temporal components cannot be used to improve performance unless the task is treated as transductive learning.
    - Following the Zindi solution, we merge the three given tables and use the Cell0 information only from the cell level table. In addition, we merge the Cell1 information since it contains information as well which might be useful in a grouped split setting.
    - Following the Zindi solution, we derive calendar features (`day`, `hour`, `weekday`) and drop the absolute timestamp.
    - We drop constant columns.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Energy",
    problem_type="regression",
    objective_metric_name="MAPE",
    group_on="BS",
    group_labels="per_sample"
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
df_train = pd.read_csv(dataset_mold.path / "ECdata.csv", parse_dates=['Time'])
# df_test = pd.read_csv("/kaggle/input/ecm-itu-zindi-kp-data/imgs_202307101549519358.csv",parse_dates=['Time'])
df_cell = pd.read_csv(dataset_mold.path / "CLdata.csv",parse_dates=['Time'])
df_bs = pd.read_csv(dataset_mold.path / "BSinfo.csv")
df_features = df_cell.merge(df_bs,on=['BS','CellName'],how='outer')
df_features = df_features[df_features['CellName']=='Cell0'].reset_index(drop=True)

df_total = df_train.merge(df_features,on=['BS','Time'],how='left')

cell1 = df_cell[df_cell['CellName']=='Cell1'].rename(columns={col: col+'_Cell1' for col in df_cell.columns if col not in ['BS','Time']})
df = df_total.merge(cell1,on=['BS','Time'],how='left')


df['day'] = df['Time'].dt.day
df['weekday_number'] = df['Time'].dt.weekday
df['hour'] = df['Time'].dt.hour

df = df.drop(columns=['Time'])

# Drop constant columns that are not useful for modeling
df = df.drop(columns=['CellName', 'ESMode4', 'CellName_Cell1', 'ESMode4_Cell1', 'ESMode5_Cell1'])

cat_cols = ["BS", "RUType", "Mode"]
for col in cat_cols:
    df[col] = df[col].astype('category')

print(df.shape)

(92629, 22)


In [3]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,BS,Energy,load,ESMode1,ESMode2,ESMode3,ESMode5,ESMode6,RUType,Mode,Frequency,Bandwidth,Antennas,TXpower,load_Cell1,ESMode1_Cell1,ESMode2_Cell1,ESMode3_Cell1,ESMode6_Cell1,day,weekday_number,hour
0,B_0,64.275037,0.487936,0.0,0.0,0.0,0.0,0.0,Type1,Mode2,365.0,20,4,6.875934,NaN,NaN,NaN,NaN,NaN,1,6,1
1,B_0,55.904335,0.344468,0.0,0.0,0.0,0.0,0.0,Type1,Mode2,365.0,20,4,6.875934,NaN,NaN,NaN,NaN,NaN,1,6,2
2,B_0,57.698057,0.193766,0.0,0.0,0.0,0.0,0.0,Type1,Mode2,365.0,20,4,6.875934,NaN,NaN,NaN,NaN,NaN,1,6,3
3,B_0,55.156951,0.222383,0.0,0.0,0.0,0.0,0.0,Type1,Mode2,365.0,20,4,6.875934,NaN,NaN,NaN,NaN,NaN,1,6,4
4,B_0,56.053812,0.175436,0.0,0.0,0.0,0.0,0.0,Type1,Mode2,365.0,20,4,6.875934,NaN,NaN,NaN,NaN,NaN,1,6,5


## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    target_feature=task_mold.target_column_name,
    problem_type=task_mold.problem_type,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 92,629
Columns: 22
Use sampling: False (sample size: 92,629)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['load', 'load_Cell1', 'ESMode6_Cell1', 'ESMode6', 'ESMode2', 'BS', 'ESMode1', 'ESMode3', 'ESMode1_Cell1', 'TXpower']
Rows remaining as candidates after top-10 filter: 2,106 (of 92,629)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,BS,Energy,load,ESMode1,ESMode2,ESMode3,ESMode5,ESMode6,RUType,Mode,Frequency,Bandwidth,Antennas,TXpower,load_Cell1,ESMode1_Cell1,ESMode2_Cell1,ESMode3_Cell1,ESMode6_Cell1,day,weekday_number,hour
0,B_0,64.275037,0.487936,0.0,0.0,0.0,0.0,0.0,Type1,Mode2,365.0,20,4,6.875934,NaN,NaN,NaN,NaN,NaN,1,6,1
1,B_0,55.904335,0.344468,0.0,0.0,0.0,0.0,0.0,Type1,Mode2,365.0,20,4,6.875934,NaN,NaN,NaN,NaN,NaN,1,6,2
2,B_0,57.698057,0.193766,0.0,0.0,0.0,0.0,0.0,Type1,Mode2,365.0,20,4,6.875934,NaN,NaN,NaN,NaN,NaN,1,6,3
3,B_0,55.156951,0.222383,0.0,0.0,0.0,0.0,0.0,Type1,Mode2,365.0,20,4,6.875934,NaN,NaN,NaN,NaN,NaN,1,6,4
4,B_0,56.053812,0.175436,0.0,0.0,0.0,0.0,0.0,Type1,Mode2,365.0,20,4,6.875934,NaN,NaN,NaN,NaN,NaN,1,6,5


In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,BS,category,0.0,0.0,923.0,"B_583, B_669, B_261, B_728, B_512, B_26, B_168, B_777, B_160, B_733"
1,RUType,category,0.0,0.0,12.0,"Type4, Type6, Type1, Type7, Type3, Type2, Type5, Type9, Type10, Type8"
2,Mode,category,0.0,0.0,2.0,"Mode2, Mode1"
3,load_Cell1,float64,87254.0,94.2,4392.0,"0.0459, 0.0458, 0.0459, 0.0344, 0.046, 0.0462, 0.0462, 0.0459, 0.046, 0.0582"
4,ESMode1_Cell1,float64,87254.0,94.2,25.0,"0.0, 1.0, 0.5194, 0.7639, 0.5444, 0.0208, 0.6972, 0.6472, 0.8125, 0.6375"
5,ESMode2_Cell1,float64,87254.0,94.2,24.0,"0.0, 0.5178, 0.5439, 0.7628, 0.81, 0.0203, 0.6958, 0.6472, 0.6356, 0.9608"
6,ESMode3_Cell1,float64,87254.0,94.2,21.0,"0.0, 0.0319, 0.0568, 0.0475, 0.0368, 0.0347, 0.0241, 0.0446, 0.0574, 0.0479"
7,ESMode6_Cell1,float64,87254.0,94.2,2705.0,"0.0, 0.7819, 0.7002, 0.6011, 0.7746, 0.7862, 0.8232, 0.8882, 0.8112, 0.7469"
8,Energy,float64,0.0,0.0,612.0,"17.9372, 17.7877, 18.2362, 18.3857, 18.0867, 18.8341, 18.5351, 17.6383, 18.9836, 18.6846"
9,load,float64,0.0,0.0,56889.0,"0.0083, 0.0082, 0.0459, 0.0083, 0.0083, 0.0083, 0.0083, 0.0083, 0.0083, 0.0082"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Energy,92629.0,28.138997,13.934645,0.747384,100.000000
load,92629.0,0.249718,0.240476,0.000000,0.993957
ESMode1,92629.0,0.041080,0.192014,0.000000,1.000000
ESMode2,92629.0,0.039769,0.189299,0.000000,1.000000
ESMode3,92629.0,0.000079,0.002384,0.000000,0.154563
ESMode5,92629.0,0.000015,0.003263,0.000000,0.768070
ESMode6,92629.0,0.009672,0.081137,0.000000,0.931032
Frequency,92629.0,368.460920,138.170799,155.600000,979.998000
Bandwidth,92629.0,16.541094,5.056101,2.000000,20.000000
Antennas,92629.0,2.187242,2.088378,1.000000,64.000000


In [8]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column rank                     
BS     1     B_583    130   0.14
       2     B_669    128   0.14
       3     B_261    128   0.14
       4     B_728    128   0.14
       5     B_512    126   0.14
Mode   1     Mode2  91994  99.31
       2     Mode1    635   0.69
RUType 1     Type4  25677  27.72
       2     Type6  22083  23.84
       3     Type1  19902  21.49
       4     Type7  12641  13.65
       5     Type3   4442   4.80

In [9]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,1.087,-0.072,194.174,0.237,log,803562.7,726880.2,lognormal


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for grouped data.",
    splits=splits,
)

Using label-per-sample grouped splits.
Repeat 0, Fold 0:
            Train N: 61406, Test N: 31223
            Target Distribution:
            	Train target distribution: 28.648860993165083
            	Test target distribution: 27.136251700568298
            Group Distribution BS:
            	Train: 615
            	Test: 308
            
Repeat 0, Fold 1:
            Train N: 62075, Test N: 30554
            Target Distribution:
            	Train target distribution: 27.920205980663358
            	Test target distribution: 28.583503264497644
            Group Distribution BS:
            	Train: 615
            	Test: 308
            
Repeat 0, Fold 2:
            Train N: 61777, Test N: 30852
            Target Distribution:
            	Train target distribution: 27.852041141368233
            	Test target distribution: 28.713587430404317
            Group Distribution BS:
            	Train: 616
            	Test: 307
            
Repeat 1, Fold 0:
            Train N: 62529, 

## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to 5g_energy_consumption/019d91e4-e573-7d82-b447-b186fcc6cdda
019d91e4-e573-7d82-b447-b186fcc6cdda
6a111b7f45a76218fa3b5c61c9fd05d1e9c964bd673a2673b932d5c60c78028a
